# 01b: MLB Statcast Incremental Data Collection

This notebook performs **incremental (daily/weekly) updates** to an existing Statcast dataset.
It is designed to be lightweight and schedulable as a Databricks job.

> **For initial full-season loads**, use `01_collect_data_and_upload_to_databricks.ipynb` (batch) instead.

### What This Notebook Does
1. Detects the most recent `game_date` already in `statcast_pitches`
2. Pulls only **new data since that date** from the Statcast API
3. Deduplicates and appends new pitches to the existing Delta table
4. Rebuilds downstream dimension tables and vector representations from the full dataset

### Tables Updated
| Table | Update Strategy |
|---|---|
| `statcast_pitches` | **Append** new rows (deduped) |
| `dim_pitchers` | **Overwrite** (rebuilt from full data) |
| `dim_batters` | **Overwrite** (rebuilt from full data) |
| `dim_players` | **Overwrite** (rebuilt from full data) |
| `dim_pitcher_arsenal` | **Overwrite** (rebuilt from full data) |
| `pitcher_vectors_median` / `pitcher_vectors_mean` | **Overwrite** (rebuilt from full data) |
| `batter_vectors_median` / `batter_vectors_mean` | **Overwrite** (rebuilt from full data) |
| `dim_batter_team_year` | **Overwrite** (rebuilt from full data) |

### Why Rebuild Downstream Tables?
Dimension tables and vectors compute aggregates (medians, means, min-max scaling) across
all data for a season. Adding new pitches changes these aggregates, so they must be
recomputed from the full `statcast_pitches` table.

### When to Use This vs. 01a
| Scenario | Use 01a (Batch) | Use This (01b) |
|---|---|---|
| First time setup | Yes | No |
| Add a new season | Yes | No |
| Daily/weekly refresh | No | Yes |
| Mid-season catch-up | Either | Yes (preferred) |

In [ ]:
%pip install pybaseball scikit-learn
dbutils.library.restartPython()

In [ ]:
import json
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
from pybaseball import statcast, playerid_reverse_lookup
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Configuration

Loads from `config/atbat_assistant.json` (created by `00_setup.ipynb`).
You can also override `LOOKBACK_DAYS` to control how far back to pull.

In [ ]:
# Load config if available, otherwise use defaults
config_path = Path("config/atbat_assistant.json")
if config_path.exists():
    CONFIG = json.loads(config_path.read_text())
    catalog = CONFIG["workspace"]["catalog"]
    schema = CONFIG["workspace"]["schema"]
    seasons = CONFIG.get("data_collection", {}).get("seasons", [2024, 2025])
    print(f"Loaded config from {config_path}")
else:
    catalog = "users"
    schema = "<your_schema>"  # Replace with your Databricks schema
    seasons = [2024, 2025]
    print("Using default config (config file not found)")

# How many extra days to overlap when pulling (handles Statcast API lag)
LOOKBACK_BUFFER_DAYS = 2

# Maximum number of days to pull in a single run (safety limit)
MAX_INCREMENTAL_DAYS = 60

# Chunk size for Statcast API calls
CHUNK_DAYS = 7

# Databricks compute
use_serverless = True
cluster_id = None

print(f"Target: {catalog}.{schema}")
print(f"Seasons tracked: {seasons}")
print(f"Lookback buffer: {LOOKBACK_BUFFER_DAYS} days")
print(f"Max incremental pull: {MAX_INCREMENTAL_DAYS} days")

In [ ]:
# Initialize Spark session
from databricks.connect import DatabricksSession

builder = DatabricksSession.builder
if use_serverless:
    builder = builder.serverless(True)
elif cluster_id:
    builder = builder.clusterId(cluster_id)
else:
    raise ValueError("Either use_serverless must be True or cluster_id must be set")

spark = builder.getOrCreate()
print("Spark session initialized")

## Step 1: Detect Last Collection Date

Query the existing `statcast_pitches` table to find the most recent `game_date`.
The incremental pull will start from `max(game_date) - LOOKBACK_BUFFER_DAYS`
to handle late-arriving data from the Statcast API.

In [ ]:
table_name = f"{catalog}.{schema}.statcast_pitches"

try:
    max_date_row = spark.sql(f"SELECT MAX(game_date) as max_date FROM {table_name}").collect()[0]
    max_date = max_date_row["max_date"]

    if max_date is None:
        raise ValueError("Table exists but has no data")

    # Convert to string if it's a date object
    if hasattr(max_date, 'strftime'):
        max_date_str = max_date.strftime('%Y-%m-%d')
    else:
        max_date_str = str(max_date)[:10]

    max_date_dt = datetime.strptime(max_date_str, '%Y-%m-%d')

    # Start from max_date minus buffer to catch late-arriving data
    pull_start_dt = max_date_dt - timedelta(days=LOOKBACK_BUFFER_DAYS)
    pull_start_str = pull_start_dt.strftime('%Y-%m-%d')

    # Current row count
    current_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name}").collect()[0]["cnt"]

    print(f"Existing table: {table_name}")
    print(f"  Current row count: {current_count:,}")
    print(f"  Most recent game_date: {max_date_str}")
    print(f"  Pull start date (with {LOOKBACK_BUFFER_DAYS}-day buffer): {pull_start_str}")

except Exception as e:
    print(f"ERROR: Could not read existing table: {e}")
    print("\nThis notebook requires an existing statcast_pitches table.")
    print("Run 01_collect_data_and_upload_to_databricks.ipynb (batch) first.")
    dbutils.notebook.exit("No existing data found. Run batch notebook first.")

In [ ]:
# Determine the end date for the pull
today = datetime.now()
pull_end_str = today.strftime('%Y-%m-%d')

# Safety check: don't pull more than MAX_INCREMENTAL_DAYS at once
days_to_pull = (today - pull_start_dt).days
if days_to_pull > MAX_INCREMENTAL_DAYS:
    print(f"WARNING: {days_to_pull} days to pull exceeds MAX_INCREMENTAL_DAYS ({MAX_INCREMENTAL_DAYS}).")
    print(f"Consider running the batch notebook (01a) instead for large date ranges.")
    pull_start_dt = today - timedelta(days=MAX_INCREMENTAL_DAYS)
    pull_start_str = pull_start_dt.strftime('%Y-%m-%d')
    print(f"Clamped pull start to: {pull_start_str}")

# Determine which season year we're pulling
pull_season = pull_start_dt.year

print(f"\nIncremental pull plan:")
print(f"  From: {pull_start_str}")
print(f"  To:   {pull_end_str}")
print(f"  Days: {(today - pull_start_dt).days}")
print(f"  Season: {pull_season}")
print(f"  Chunk size: {CHUNK_DAYS} days")

## Step 2: Pull Incremental Data from Statcast

Pulls data in chunks and collects into a single pandas DataFrame before deduplication.

In [ ]:
all_chunks = []
current_date = pull_start_dt
final_date = today
chunk_num = 0

print(f"Pulling Statcast data from {pull_start_str} to {pull_end_str}...\n")

while current_date <= final_date:
    chunk_num += 1
    chunk_end = min(current_date + timedelta(days=CHUNK_DAYS - 1), final_date)

    chunk_start_str = current_date.strftime('%Y-%m-%d')
    chunk_end_str = chunk_end.strftime('%Y-%m-%d')

    print(f"  Chunk {chunk_num}: {chunk_start_str} to {chunk_end_str}", end=" ")

    try:
        data = statcast(start_dt=chunk_start_str, end_dt=chunk_end_str)

        if data is not None and len(data) > 0:
            data['season'] = pull_season
            all_chunks.append(data)
            print(f"-> {len(data):,} pitches")
        else:
            print("-> no data")

    except Exception as e:
        print(f"-> ERROR: {e}")

    current_date = chunk_end + timedelta(days=1)

if all_chunks:
    new_data = pd.concat(all_chunks, ignore_index=True)
    print(f"\nTotal pitches pulled: {len(new_data):,}")
else:
    print("\nNo new data found. The dataset is up to date.")
    new_data = None

## Step 3: Deduplicate & Append

Since we pull with a lookback buffer, some rows may already exist in the table.
We use a `MERGE` (upsert) pattern:
- Match on a composite key of `(game_pk, at_bat_number, pitch_number)`
- Only insert rows that don't already exist

In [ ]:
if new_data is not None and len(new_data) > 0:
    print(f"Deduplicating and appending {len(new_data):,} pitches...\n")

    # Convert to Spark DataFrame
    new_spark_df = spark.createDataFrame(new_data)

    # Schema alignment: cast columns to match existing table types
    try:
        existing_schema = {f.name: f.dataType for f in spark.table(table_name).schema.fields}
        for col_name in new_spark_df.columns:
            if col_name in existing_schema:
                current_type = dict(new_spark_df.dtypes).get(col_name)
                existing_type_str = existing_schema[col_name].simpleString()
                if current_type != existing_type_str:
                    new_spark_df = new_spark_df.withColumn(
                        col_name, F.col(col_name).cast(existing_schema[col_name])
                    )
        print("Schema aligned with existing table")
    except Exception as e:
        print(f"Warning: Schema alignment issue: {e}")

    # Register as temp view for MERGE
    new_spark_df.createOrReplaceTempView("new_pitches")

    # Count before
    count_before = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name}").collect()[0]["cnt"]

    # MERGE: insert only rows that don't already exist
    merge_sql = f"""
    MERGE INTO {table_name} AS target
    USING new_pitches AS source
    ON target.game_pk = source.game_pk
       AND target.at_bat_number = source.at_bat_number
       AND target.pitch_number = source.pitch_number
    WHEN NOT MATCHED THEN INSERT *
    """

    result = spark.sql(merge_sql)
    print("MERGE complete")

    # Count after
    count_after = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name}").collect()[0]["cnt"]
    rows_added = count_after - count_before

    print(f"\nResults:")
    print(f"  Rows before: {count_before:,}")
    print(f"  Rows after:  {count_after:,}")
    print(f"  New rows added: {rows_added:,}")
    print(f"  Duplicates skipped: {len(new_data) - rows_added:,}")

    # Update max date for reporting
    new_max = spark.sql(f"SELECT MAX(game_date) as max_date FROM {table_name}").collect()[0]["max_date"]
    print(f"  New max game_date: {new_max}")

else:
    print("No new data to append. Skipping to downstream table rebuild.")
    rows_added = 0

## Step 4: Rebuild Downstream Tables

Even if no new rows were added, you may want to rebuild downstream tables to
pick up any corrections in the source data. Set `FORCE_REBUILD = True` to
always rebuild, or leave it as `False` to skip if no rows were added.

In [ ]:
FORCE_REBUILD = False  # Set to True to always rebuild downstream tables

should_rebuild = rows_added > 0 or FORCE_REBUILD

if should_rebuild:
    print(f"Rebuilding downstream tables (rows_added={rows_added:,}, force={FORCE_REBUILD})...")
else:
    print("No new rows and FORCE_REBUILD=False. Skipping downstream rebuild.")
    print("Set FORCE_REBUILD = True to rebuild anyway.")

### 4a: Rebuild Player Dimension Tables

In [ ]:
if should_rebuild:
    # Read full statcast data
    combined_data_spark = spark.table(table_name)

    # ---- dim_pitchers ----
    print("\n--- Rebuilding dim_pitchers ---")
    pitcher_ids_spark = combined_data_spark.select('pitcher').distinct()
    pitcher_ids_pd = pitcher_ids_spark.toPandas()
    unique_pitcher_ids = pitcher_ids_pd['pitcher'].dropna().astype(int).tolist()
    print(f"  Looking up names for {len(unique_pitcher_ids):,} unique pitchers...")

    try:
        pitcher_lookup = playerid_reverse_lookup(unique_pitcher_ids, key_type='mlbam')
        pitcher_dim_pd = pitcher_lookup[['key_mlbam', 'name_first', 'name_last']].copy()
        pitcher_dim_pd = pitcher_dim_pd.rename(columns={'key_mlbam': 'player_id'})
        print(f"  Found {len(pitcher_dim_pd):,} pitcher ID mappings")
    except Exception as e:
        print(f"  Warning: playerid_reverse_lookup failed: {e}")
        pitcher_dim_pd = pd.DataFrame({'player_id': unique_pitcher_ids})
        pitcher_dim_pd['name_first'] = None
        pitcher_dim_pd['name_last'] = None

    # Infer pitcher handedness from statcast data
    pitcher_hand = (combined_data_spark
        .groupBy('pitcher')
        .agg(F.first('p_throws').alias('throws'))
        .withColumnRenamed('pitcher', 'player_id')
        .toPandas())

    pitcher_dim_pd = pitcher_dim_pd.merge(pitcher_hand, on='player_id', how='left')

    pitcher_dim_spark = spark.createDataFrame(pitcher_dim_pd)
    (pitcher_dim_spark.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{schema}.dim_pitchers"))
    print(f"  Uploaded dim_pitchers ({len(pitcher_dim_pd):,} rows)")

    # ---- dim_batters ----
    print("\n--- Rebuilding dim_batters ---")
    batter_ids_spark = combined_data_spark.select('batter').distinct()
    batter_ids_pd = batter_ids_spark.toPandas()
    unique_batter_ids = batter_ids_pd['batter'].dropna().astype(int).tolist()
    print(f"  Looking up names for {len(unique_batter_ids):,} unique batters...")

    try:
        batter_lookup = playerid_reverse_lookup(unique_batter_ids, key_type='mlbam')
        batter_dim_pd = batter_lookup[['key_mlbam', 'name_first', 'name_last']].copy()
        batter_dim_pd = batter_dim_pd.rename(columns={'key_mlbam': 'player_id'})
        print(f"  Found {len(batter_dim_pd):,} batter ID mappings")
    except Exception as e:
        print(f"  Warning: playerid_reverse_lookup failed: {e}")
        batter_dim_pd = pd.DataFrame({'player_id': unique_batter_ids})
        batter_dim_pd['name_first'] = None
        batter_dim_pd['name_last'] = None

    # Infer batter handedness from statcast data
    batter_hand = (combined_data_spark
        .groupBy('batter')
        .agg(F.first('stand').alias('bats'))
        .withColumnRenamed('batter', 'player_id')
        .toPandas())

    batter_dim_pd = batter_dim_pd.merge(batter_hand, on='player_id', how='left')

    batter_dim_spark = spark.createDataFrame(batter_dim_pd)
    (batter_dim_spark.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{schema}.dim_batters"))
    print(f"  Uploaded dim_batters ({len(batter_dim_pd):,} rows)")

    # ---- dim_players ----
    print("\n--- Rebuilding dim_players ---")
    pitchers_with_type = pitcher_dim_spark.withColumn('player_type', F.lit('pitcher')).withColumnRenamed('throws', 'hand')
    batters_with_type = batter_dim_spark.withColumn('player_type', F.lit('batter')).withColumnRenamed('bats', 'hand')

    # Ensure same columns
    common_cols = ['player_id', 'name_first', 'name_last', 'hand', 'player_type']
    dim_players = pitchers_with_type.select(*common_cols).unionByName(
        batters_with_type.select(*common_cols)
    ).dropDuplicates(['player_id', 'player_type'])

    (dim_players.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{schema}.dim_players"))
    print(f"  Uploaded dim_players ({dim_players.count():,} rows)")

    print("\nDimension tables rebuilt successfully.")

### 4b: Rebuild Pitcher Vectors & Arsenal

In [ ]:
if should_rebuild:
    from pyspark.sql.window import Window
    from sklearn.preprocessing import MinMaxScaler
    import numpy as np

    combined_data_spark = spark.table(table_name)

    # ---- Pitcher Vectors ----
    print("\n--- Rebuilding pitcher vectors ---")

    pitcher_features = [
        'release_speed', 'release_spin_rate', 'release_pos_x', 'release_pos_y',
        'release_pos_z', 'release_extension', 'pfx_x', 'pfx_z', 'vx0', 'vy0',
        'vz0', 'ax', 'ay', 'az', 'effective_speed', 'arm_angle'
    ]
    print(f"  Features ({len(pitcher_features)}): {', '.join(pitcher_features)}")

    def create_pitcher_vectors(stat_type='median'):
        # Filter to non-null pitches with required features
        base_df = combined_data_spark.filter(
            F.col('pitcher').isNotNull() &
            F.col('pitch_type').isNotNull() &
            F.col('season').isNotNull()
        )

        # Count pitches per pitcher-season-pitch_type
        pitch_counts = base_df.groupBy('pitcher', 'season', 'pitch_type').count()
        qualified = pitch_counts.filter(F.col('count') >= 25)

        # Join back to filter
        filtered = base_df.join(
            qualified.select('pitcher', 'season', 'pitch_type'),
            on=['pitcher', 'season', 'pitch_type'],
            how='inner'
        )

        # Winsorize: clip to 1st and 99th percentile per feature
        for feat in pitcher_features:
            pcts = filtered.approxQuantile(feat, [0.01, 0.99], 0.01)
            if len(pcts) == 2:
                filtered = filtered.withColumn(
                    feat, F.when(F.col(feat) < pcts[0], pcts[0])
                          .when(F.col(feat) > pcts[1], pcts[1])
                          .otherwise(F.col(feat))
                )

        # Aggregate
        agg_exprs = []
        for feat in pitcher_features:
            if stat_type == 'median':
                agg_exprs.append(F.percentile_approx(feat, 0.5).alias(feat))
            else:
                agg_exprs.append(F.avg(feat).alias(feat))

        agg_exprs.append(F.first('p_throws').alias('p_throws'))

        agg_df = filtered.groupBy('pitcher', 'season', 'pitch_type').agg(*agg_exprs)
        agg_df = agg_df.withColumnRenamed('pitcher', 'player_id')

        # Min-max scale
        agg_pd = agg_df.toPandas()

        if len(agg_pd) > 0:
            scaler = MinMaxScaler()
            scaled_values = scaler.fit_transform(agg_pd[pitcher_features].fillna(0))
            scaled_values = np.clip(scaled_values, 0, 1)

            agg_pd['embedding_vector'] = [row.tolist() for row in scaled_values]
            print(f"  {stat_type.upper()} vectors: {len(agg_pd):,} rows")
        else:
            agg_pd['embedding_vector'] = []
            print(f"  {stat_type.upper()} vectors: 0 rows (no qualifying data)")

        return spark.createDataFrame(agg_pd)

    pitcher_vectors_median = create_pitcher_vectors('median')
    pitcher_vectors_mean = create_pitcher_vectors('mean')

    # Join with dim_pitchers for names
    pitcher_names = spark.table(f"{catalog}.{schema}.dim_pitchers").select('player_id', 'name_first', 'name_last')
    pitcher_vectors_median = pitcher_vectors_median.join(pitcher_names, on='player_id', how='left')
    pitcher_vectors_mean = pitcher_vectors_mean.join(pitcher_names, on='player_id', how='left')

    (pitcher_vectors_median.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{schema}.pitcher_vectors_median"))
    print(f"  Uploaded pitcher_vectors_median")

    (pitcher_vectors_mean.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{schema}.pitcher_vectors_mean"))
    print(f"  Uploaded pitcher_vectors_mean")

    # ---- dim_pitcher_arsenal ----
    print("\n--- Rebuilding dim_pitcher_arsenal ---")
    arsenal = (combined_data_spark
        .filter(F.col('pitch_type').isNotNull())
        .groupBy('pitcher', 'season', 'pitch_type', 'p_throws')
        .agg(
            F.count('*').alias('pitch_count'),
            F.avg('release_speed').alias('avg_speed'),
            F.avg('release_spin_rate').alias('avg_spin')
        )
        .withColumnRenamed('pitcher', 'player_id'))

    # Add usage percentage
    total_window = Window.partitionBy('player_id', 'season')
    arsenal = arsenal.withColumn(
        'usage_pct',
        F.col('pitch_count') / F.sum('pitch_count').over(total_window)
    )

    (arsenal.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{schema}.dim_pitcher_arsenal"))
    print(f"  Uploaded dim_pitcher_arsenal ({arsenal.count():,} rows)")

    print("\nPitcher vectors and arsenal rebuilt successfully.")

### 4c: Rebuild Batter Vectors

In [ ]:
if should_rebuild:
    combined_data_spark = spark.table(table_name)

    print("\n--- Rebuilding batter vectors ---")

    batter_features = [
        'launch_speed', 'launch_angle', 'hit_distance_sc',
        'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle',
        'woba_value', 'woba_denom', 'babip_value', 'iso_value',
        'launch_speed_angle', 'barrel'
    ]
    print(f"  Features ({len(batter_features)}): {', '.join(batter_features)}")

    def create_batter_vectors(stat_type='median'):
        base_df = combined_data_spark.filter(
            F.col('batter').isNotNull() &
            F.col('pitch_type').isNotNull() &
            F.col('season').isNotNull()
        )

        pitch_counts = base_df.groupBy('batter', 'season', 'pitch_type').count()
        qualified = pitch_counts.filter(F.col('count') >= 25)

        filtered = base_df.join(
            qualified.select('batter', 'season', 'pitch_type'),
            on=['batter', 'season', 'pitch_type'],
            how='inner'
        )

        # Winsorize
        for feat in batter_features:
            pcts = filtered.approxQuantile(feat, [0.01, 0.99], 0.01)
            if len(pcts) == 2:
                filtered = filtered.withColumn(
                    feat, F.when(F.col(feat) < pcts[0], pcts[0])
                          .when(F.col(feat) > pcts[1], pcts[1])
                          .otherwise(F.col(feat))
                )

        # Aggregate
        agg_exprs = []
        for feat in batter_features:
            if stat_type == 'median':
                agg_exprs.append(F.percentile_approx(feat, 0.5).alias(feat))
            else:
                agg_exprs.append(F.avg(feat).alias(feat))

        agg_exprs.append(F.first('stand').alias('stand'))

        agg_df = filtered.groupBy('batter', 'season', 'pitch_type').agg(*agg_exprs)
        agg_df = agg_df.withColumnRenamed('batter', 'player_id')

        # Min-max scale
        agg_pd = agg_df.toPandas()

        if len(agg_pd) > 0:
            scaler = MinMaxScaler()
            scaled_values = scaler.fit_transform(agg_pd[batter_features].fillna(0))
            scaled_values = np.clip(scaled_values, 0, 1)

            agg_pd['embedding_vector'] = [row.tolist() for row in scaled_values]
            print(f"  {stat_type.upper()} vectors: {len(agg_pd):,} rows")
        else:
            agg_pd['embedding_vector'] = []
            print(f"  {stat_type.upper()} vectors: 0 rows (no qualifying data)")

        return spark.createDataFrame(agg_pd)

    batter_vectors_median = create_batter_vectors('median')
    batter_vectors_mean = create_batter_vectors('mean')

    # Join with dim_batters for names
    batter_names = spark.table(f"{catalog}.{schema}.dim_batters").select('player_id', 'name_first', 'name_last')
    batter_vectors_median = batter_vectors_median.join(batter_names, on='player_id', how='left')
    batter_vectors_mean = batter_vectors_mean.join(batter_names, on='player_id', how='left')

    (batter_vectors_median.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{schema}.batter_vectors_median"))
    print(f"  Uploaded batter_vectors_median")

    (batter_vectors_mean.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{schema}.batter_vectors_mean"))
    print(f"  Uploaded batter_vectors_mean")

    print("\nBatter vectors rebuilt successfully.")

### 4d: Rebuild Batter-Team-Year Dimension

In [ ]:
if should_rebuild:
    from pybaseball import batting_stats, pitching_stats

    print("\n--- Rebuilding dim_batter_team_year ---")

    combined_data_spark = spark.table(table_name)

    def _normalize_name_expr(col):
        return F.lower(F.regexp_replace(F.trim(col), r"[.'-]", ""))

    all_dim_dfs = []

    for season in seasons:
        print(f"  Processing season {season}...")
        try:
            # Get team info from batting stats
            bat = batting_stats(season, qual=1)

            if bat is not None and len(bat) > 0:
                bat_pd = bat[['Name', 'Team']].copy()
                bat_pd['season'] = season

                # Normalize names for matching
                bat_pd['first_norm'] = bat_pd['Name'].apply(lambda x: x.split()[0].lower().replace('.', '').replace("'", '').replace('-', '') if isinstance(x, str) and ' ' in x else '')
                bat_pd['last_norm'] = bat_pd['Name'].apply(lambda x: ' '.join(x.split()[1:]).lower().replace('.', '').replace("'", '').replace('-', '') if isinstance(x, str) and ' ' in x else '')

                bat_spark = spark.createDataFrame(bat_pd)

                # Match to player IDs via dim_batters
                dim_b = spark.table(f"{catalog}.{schema}.dim_batters")
                dim_b = dim_b.withColumn('first_norm', _normalize_name_expr('name_first'))
                dim_b = dim_b.withColumn('last_norm', _normalize_name_expr('name_last'))

                matched = bat_spark.join(
                    dim_b.select('player_id', F.col('first_norm').alias('db_first'), F.col('last_norm').alias('db_last'), 'bats'),
                    (bat_spark['first_norm'] == dim_b['first_norm']) & (bat_spark['last_norm'] == dim_b['last_norm']),
                    how='inner'
                ).select('player_id', 'Name', 'Team', 'season', 'bats')

                matched = matched.withColumnRenamed('Name', 'player_name').withColumnRenamed('Team', 'team')
                all_dim_dfs.append(matched)
                print(f"    Matched {matched.count():,} batters")
            else:
                print(f"    No batting stats available for {season}")

        except Exception as e:
            print(f"    Error processing season {season}: {e}")

    if all_dim_dfs:
        from functools import reduce
        dim_batter_team = reduce(lambda a, b: a.unionByName(b), all_dim_dfs)
        dim_batter_team = dim_batter_team.dropDuplicates(['player_id', 'season'])

        (dim_batter_team.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"{catalog}.{schema}.dim_batter_team_year"))
        print(f"  Uploaded dim_batter_team_year ({dim_batter_team.count():,} rows)")
    else:
        print("  Warning: No batter-team-year data generated")

    print("\nBatter-team-year dimension rebuilt successfully.")

## Summary

In [ ]:
print("="*70)
print("INCREMENTAL DATA COLLECTION COMPLETE")
print("="*70)

final_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name}").collect()[0]["cnt"]
final_max = spark.sql(f"SELECT MAX(game_date) as max_date FROM {table_name}").collect()[0]["max_date"]

print(f"\nstatcast_pitches:")
print(f"  Total rows: {final_count:,}")
print(f"  Latest date: {final_max}")
print(f"  New rows added this run: {rows_added:,}")
print(f"  Downstream tables rebuilt: {should_rebuild}")

if should_rebuild:
    tables_rebuilt = [
        'dim_pitchers', 'dim_batters', 'dim_players',
        'pitcher_vectors_median', 'pitcher_vectors_mean',
        'dim_pitcher_arsenal',
        'batter_vectors_median', 'batter_vectors_mean',
        'dim_batter_team_year',
    ]
    print(f"\nTables rebuilt:")
    for t in tables_rebuilt:
        try:
            cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {catalog}.{schema}.{t}").collect()[0]["cnt"]
            print(f"  {t}: {cnt:,} rows")
        except:
            print(f"  {t}: (not found)")

print(f"\nDone! Schedule this notebook to run daily/weekly via a Databricks job.")